# Najafi et al. (2020) on DANDI:000016, part 2: choice decoders in expert sessions

Paper doi:10.1016/j.neuron.2019.09.045; dataset DANDI:000016 version 0.260915.2309, the only
published version (the paper cites none; see `01_single_neurons_correlations.ipynb` for the
data checks). Claims file `claims/najafi-2020.yaml`.

Decoders as in the claims file's opening comment (linear SVM per 97 ms bin, z-scored neurons,
choices matched, 50 random 90/10 splits, C by minimum mean test error, shuffled-label control),
at the bin 0-97 ms before the choice, on correct trials, every session with at least 10 correct
trials of each choice. Two departures for runtime, recorded as deviations: C is chosen from
10^-6 to 10^6 in steps of 1 (the paper's grid has steps of 0.2), and the excitatory, mixed and
2n-excitatory populations use 5 random subsamples (the paper 50). A check of the full grid on a
few sessions is at the end.

| Claim | Here |
|---|---|
| fig3bc-decoding-similar | tested |
| fig3hi-mixed-no-advantage | tested |
| fig3df-weights-similar | tested (bin -1 only) |
| figs6de-incorrect-trials | tested |
| fig4bd-stability-similar | tested |
| fig7-noise-correlations-limit-decoding | tested |
| figs12-licking-running | blocked (lick spout not identified); the running part is shown |
| learning decoders | `03_learning_decoders.ipynb` |

In [ ]:
import time, resource, os
import numpy as np
import pandas as pd
from scipy import stats
from joblib import Parallel, delayed
import najafi as nj

T0 = time.time()
A = nj.assets()
S = nj.load_all(A, workers=6, alignments=(nj.CHOICE,), behaviour=True)
for s in S:  # keep only what the decoders use
    s["al"][nj.CHOICE].pop("frames", None)
print("loaded", len(S), "sessions in %.0f s, peak memory %.1f GB" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
D = pd.DataFrame([{"mouse": s["mouse"], "session": s["session_id"], "correct": int(nj.trial_mask(s, (1,)).sum()),
                   "incorrect": int(nj.trial_mask(s, (0,)).sum()), "exc": int((s["type"] == "excitatory").sum()),
                   "inh": int((s["type"] == "inhibitory").sum()), "all": len(s["type"]),
                   "minchoice_corr": int(min(np.sum(s["trials"].choice.to_numpy()[nj.trial_mask(s, (1,))] == v) for v in (0, 1)))} for s in S])
print(D.groupby("mouse")[["correct", "incorrect", "exc", "inh", "all", "minchoice_corr"]].median())
print("sessions with fewer than 10 correct trials of a choice:", int((D.minchoice_corr < 10).sum()))
print("sessions with fewer than twice as many excitatory as inhibitory neurons:", int((D.exc < 2 * D.inh).sum()))
wh = np.concatenate([s["wheel"].ravel() for s in S]); wh = wh[~np.isnan(wh)]
print("wheel (revolutions, cumulative, in the 200 ms before the choice): %.3f to %.3f; samples every %.3f s" % (wh.min(), wh.max(), np.diff(S[0]["wheel_t"]).mean()))
print("lick trace values:", np.unique(np.concatenate([np.unique(s["lick"][~np.isnan(s["lick"])]) for s in S])), "time range of the lick trace, s:", S[0]["lick_range"])

In [ ]:
# one generator per session, seeded from its index, so a session's numbers do not depend on the others
os.environ.setdefault("OMP_NUM_THREADS", "1")
t1 = time.time()
OUT = Parallel(n_jobs=4)(delayed(nj.decode_session)(s, 1000 + i) for i, s in enumerate(S))
print("decoders: %.1f min" % ((time.time() - t1) / 60))
OUT = [o for o in OUT if "all" in o]
print("sessions decoded:", len(OUT), pd.Series([o["mouse"] for o in OUT]).value_counts().sort_index().to_dict())
Q = pd.DataFrame([{"mouse": o["mouse"], "session": o["session"], "nI": o["nI"], "ntr": o["ntr"],
                   "all": o["all"]["acc"], "all_sh": o["all"]["acc_sh"], "inh": o["inh"]["acc"], "inh_sh": o["inh"]["acc_sh"],
                   "exc": np.mean([x["acc"] for x in o["exc_subs"]]), "exc_sh": np.mean([x["acc_sh"] for x in o["exc_subs"]]),
                   "all_inc": o["all"]["acc_inc"], "inh_inc": o["inh"]["acc_inc"], "exc_inc": np.mean([x["acc_inc"] for x in o["exc_subs"]]),
                   "all_inc_sh": o["all"]["acc_inc_sh"], "inh_inc_sh": o["inh"]["acc_inc_sh"], "exc_inc_sh": np.mean([x["acc_inc_sh"] for x in o["exc_subs"]]),
                   "n_inc": o["n_incorr"], "mixed": np.mean(o["mixed"]) if "mixed" in o else np.nan, "exc2n": np.mean(o["exc2n"]) if "exc2n" in o else np.nan,
                   "pseudo": o["pseudo"]["acc"], "all_alltr": o.get("all_alltr", {}).get("acc", np.nan), "pseudo_alltr": o.get("pseudo_alltr", {}).get("acc", np.nan)} for o in OUT])
print("accuracy range over sessions (percent): all %.1f to %.1f, inh %.1f to %.1f, exc %.1f to %.1f" % (Q["all"].min(), Q["all"].max(), Q.inh.min(), Q.inh.max(), Q.exc.min(), Q.exc.max()))

def ttest(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return stats.ttest_ind(a[~np.isnan(a)], b[~np.isnan(b)])

## fig3bc-decoding-similar

> Choice can be decoded from population activity well above chance, and inhibitory populations
> decode it as accurately as excitatory populations of the same size, in every mouse.

In [ ]:
for m, g in Q.groupby("mouse"):
    pe, pi, pa = ttest(g.exc, g.exc_sh).pvalue, ttest(g.inh, g.inh_sh).pvalue, ttest(g["all"], g.all_sh).pvalue
    d = ttest(g.inh, g.exc)
    print(f"mouse {m} ({len(g)} sessions): all {g['all'].mean():.1f} (shuffle {g.all_sh.mean():.1f}, p = {pa:.2g}); "
          f"exc {g.exc.mean():.1f} (shuffle {g.exc_sh.mean():.1f}, p = {pe:.2g}); inh {g.inh.mean():.1f} (shuffle {g.inh_sh.mean():.1f}, p = {pi:.2g}); "
          f"inh - exc {g.inh.mean()-g.exc.mean():+.2f}, t = {d.statistic:.2f}, p = {d.pvalue:.3g}")

**Result: reproduced.** In every mouse the inhibitory and subsampled excitatory decoders are above their shuffles (p below 0.003), and they do not differ: mean accuracies E 65.2, 64.4, 60.1, 67.3 and I 64.7, 65.6, 58.8, 69.7 percent (p = 0.71, 0.35, 0.66, 0.15). The all-neuron decoder reaches 73.3 to 87.0 percent.

## fig3hi-mixed-no-advantage

> A population of half excitatory and half inhibitory neurons decodes choice with accuracy
> similar to that of an excitatory population of the same size, so a mixed population offers no
> major advantage.

In [ ]:
print("sessions without 2n excitatory neurons (left out):", int(Q.mixed.isna().sum()))
for m, g in Q.dropna(subset=["mixed"]).groupby("mouse"):
    d = ttest(g.mixed, g.exc2n)
    print(f"mouse {m} ({len(g)} sessions): mixed {g.mixed.mean():.1f}, 2n excitatory {g.exc2n.mean():.1f}, difference {g.mixed.mean()-g.exc2n.mean():+.2f}, t = {d.statistic:.2f}, p = {d.pvalue:.3g}")

**Result: reproduced.** Mixed populations (n inhibitory plus n excitatory) and 2n-excitatory populations decode the choice equally in every mouse (differences -0.15 to +0.71 points, p 0.69 to 0.97).

## fig3df-weights-similar

> In the decoder trained on all neurons, excitatory and inhibitory neurons receive weights of
> matched magnitude throughout the trial and with very similar distributions.

In [ ]:
nE = sum(len(o["w_E"]) for o in OUT); nI = sum(len(o["w_I"]) for o in OUT)
print(f"neurons pooled over decoded sessions: excitatory {nE} (expected 42019 +/- 4202), inhibitory {nI} (expected 5172 +/- 517)")
W = pd.DataFrame([{"mouse": o["mouse"], "E": o["w_E"].mean(), "I": o["w_I"].mean()} for o in OUT])
nsig = 0
for m, g in W.groupby("mouse"):
    d = ttest(g.E, g.I); nsig += d.pvalue < 0.05
    pool = stats.ttest_ind(np.concatenate([o["w_E"] for o in OUT if o["mouse"] == m]), np.concatenate([o["w_I"] for o in OUT if o["mouse"] == m]))
    print(f"mouse {m}: mean |w| E {g.E.mean():.4f}, I {g.I.mean():.4f}, I - E {g.I.mean()-g.E.mean():+.4f}, across sessions p = {d.pvalue:.3g}; [neurons pooled p = {pool.pvalue:.3g}]")
print("mice with a significant difference across sessions:", nsig)
allw = np.concatenate([np.concatenate([o["w_E"], o["w_I"]]) for o in OUT])
print("range of |w| (unit-length weight vectors): %.2g to %.3f" % (allw.min(), allw.max()))

**Result: reproduced.** In the all-neuron decoder at 0-97 ms before the choice, the mean |w| of unit-length weight vectors does not differ between types in any mouse across sessions (p = 0.68, 0.079, 0.32, 0.13). The pooled counts, 42019 excitatory and 5172 inhibitory neurons, equal the expected values. Weights were computed at this bin only, not through the trial.

## figs6de-incorrect-trials

> Decoders trained on correct trials predict the choice on incorrect trials with high accuracy,
> slightly lower than on correct trials, and the reduction is similar for excitatory and
> inhibitory populations in every mouse.

In [ ]:
for minn, lab in ((1, "sessions with at least one incorrect trial of each choice"), (10, "variant, at least 10 incorrect trials of each choice")):
    G = Q[[min(n) >= minn for n in Q.n_inc]]
    print(lab, ":", len(G), "sessions", G.groupby("mouse").size().to_dict())
    red = {}
    for m, g in G.groupby("mouse"):
        out = []
        for pop in ("all", "exc", "inh"):
            p = ttest(g[pop + "_inc"], g[pop + "_inc_sh"]).pvalue
            out.append(f"{pop} incorrect {g[pop+'_inc'].mean():.1f} (shuffle {g[pop+'_inc_sh'].mean():.1f}, p = {p:.2g}), correct {g[pop].mean():.1f}")
        rE, rI = g.exc - g.exc_inc, g.inh - g.inh_inc
        d = ttest(rI, rE); red[m] = (g["all"] - g.all_inc).mean()
        print(f"  mouse {m}: " + "; ".join(out) + f"; reduction exc {rE.mean():.1f}, inh {rI.mean():.1f}, p = {d.pvalue:.3g}")
    print("  reduction of the all-neuron decoder by mouse:", {k: round(v, 1) for k, v in red.items()}, "largest in mouse", max(red, key=red.get))

**Result: partially reproduced.** Decoders trained on correct trials predict the choice on incorrect trials above the shuffle in mice 1 and 2 for all three populations (p below 3e-6). In mice 3 and 4 they do not (mouse 3 p 0.12 to 0.69; mouse 4 accuracy 44 to 47 percent, below the shuffle). The reduction from correct to incorrect trials does not differ between inhibitory and excitatory decoders in any mouse (p 0.31 to 0.95). It is largest in mouse 4 for the all-neuron decoder (43.0 points, others 8.4 to 8.6). The variant with at least 10 incorrect trials of each choice gives the same pattern.

## fig4bd-stability-similar

> Choice decoders lose accuracy at a similar rate away from their training time for excitatory
> and inhibitory populations, and their stability duration is indistinguishable.

In [ ]:
rows = []
for o in OUT:
    for pop, recs in (("inh", [o["inh"]]), ("exc", o["exc_subs"])):
        for rule in ("sd", "se"):
            for cont, ci in (("any", 0), ("contig", 1)):
                v = [r["stab"][rule][ci] * 97 for r in recs]
                pt = [r["stab"]["p_train"] for r in recs]
                rows.append({"mouse": o["mouse"], "session": o["session"], "pop": pop, "rule": rule, "cont": cont, "dur": np.mean(v),
                             "p_train": np.max(pt) if pop == "inh" else np.median(pt), "nbins": len(o["k"])})
ST = pd.DataFrame(rows)
for alpha in (0.001, 0.05):
    for rule in ("sd", "se"):
        for cont in ("any", "contig"):
            g = ST[(ST.rule == rule) & (ST.cont == cont)]
            ok = g.pivot_table(index="session", columns="pop", values="p_train").max(1) < alpha
            g = g[g.session.isin(ok[ok].index)]
            mm = g.groupby(["mouse", "pop"]).dur.mean().unstack()
            d = stats.ttest_ind(mm.exc, mm.inh)
            per = {m: round(stats.ttest_ind(x[x["pop"] == "exc"].dur, x[x["pop"] == "inh"].dur).pvalue, 3) for m, x in g.groupby("mouse")}
            print(f"alpha {alpha}, {rule.upper()} rule, {cont} bins: {ok.sum()} sessions; mouse means exc {mm.exc.round(0).to_dict()} inh {mm.inh.round(0).to_dict()} ms; "
                  f"across mice t = {d.statistic:.2f}, p = {d.pvalue:.3g}; per mouse across sessions p {per}")
sa = pd.DataFrame([{"mouse": o["mouse"], "k": k, "inh": a, "exc": e} for o in OUT for k, a, e in zip(o["k"], o["inh"]["stab_acc"], np.mean([x["stab_acc"] for x in o["exc_subs"]], 0))])
print("accuracy of the decoder trained at bin -1, tested at bins -10, -5, -1, 0, 3 (mean over sessions):")
print(sa[sa.k.isin([-10, -5, -1, 0, 3])].groupby(["mouse", "k"])[["exc", "inh"]].mean().round(1).unstack().to_string())

**Result: reproduced.** The stability duration of decoders trained at 0-97 ms before the choice does not differ between excitatory and inhibitory populations by the t-test of the mouse means under every reading (SD or SE rule, any or contiguous bins, alpha 0.001 or 0.05; p 0.48 to 0.98). Per mouse across sessions, no test is significant either (smallest p 0.10).

## fig7-noise-correlations-limit-decoding

> Removing noise correlations by building pseudo populations increases choice decoding accuracy
> of the full population in every mouse.

In [ ]:
for real, ps, lab in (("all", "pseudo", "correct trials"), ("all_alltr", "pseudo_alltr", "all trials with a choice")):
    print(lab)
    for m, g in Q.dropna(subset=[real, ps]).groupby("mouse"):
        ch = g[ps] - g[real]
        p1 = stats.ttest_1samp(ch, 0).pvalue; p2 = ttest(g[ps], g[real]).pvalue
        print(f"  mouse {m} ({len(g)} sessions): real {g[real].mean():.1f}, pseudo {g[ps].mean():.1f}, change {ch.mean():+.2f} (range {ch.min():+.1f} to {ch.max():+.1f}); "
              f"(a) one-sample p = {p1:.2g}; (b) unpaired p = {p2:.2g}")

**Result: reproduced, on an assumption.** Pseudo populations raise the all-neuron accuracy in every mouse on correct trials, by a mean of +5.38, +5.77, +4.32 and +5.44 points. Reading (a), the one-sample test of the per-session changes, is significant in every mouse (p = 5.3e-10, 1.5e-13, 0.011, 5e-11). Reading (b), the unpaired test, fails in mouse 3 (p = 0.32, 7 sessions). Reading (a) is adopted as an assumption. With all trials with a choice, the changes are +4.9 to +6.7 points (mouse 3 p = 0.08 by reading (a)).

## figs12-licking-running (running part only)

> Licking and running do not explain the learning-related changes; centre-spout licks before
> left and right choices are similar early and late in training, and decoding remains accurate in
> sessions where running did not differ between choices.

The lick trace in the files (`firstSideTryAl_lick`, 0 or 1 per ms) does not say which spout was
licked, so the centre-spout licks of the criterion cannot be separated; the claim is blocked.
The running part is shown.

In [ ]:
rr = []
for o in OUT:
    if "run" in o and o["mouse"] in nj.LEARNING_MICE:
        d, c = o["run"]
        p = stats.ttest_ind(d[c == 0], d[c == 1]).pvalue if min((c == 0).sum(), (c == 1).sum()) > 1 else np.nan
        rr.append({"mouse": o["mouse"], "p_run": p, "all": o["all"]["acc"], "all_sh": o["all"]["acc_sh"]})
RR = pd.DataFrame(rr)
for m, g in RR.groupby("mouse"):
    g2 = g[g.p_run > 0.05]
    print(f"mouse {m}: {len(g2)} of {len(g)} sessions with running not different (p > 0.05); all-neuron accuracy there {g2['all'].mean():.1f} against shuffle {g2.all_sh.mean():.1f}, p = {ttest(g2['all'], g2.all_sh).pvalue:.2g}")

## Check of the coarser C grid

On four sessions (one per mouse), the all-neuron and inhibitory decoders with the paper's grid
(steps of 0.2) against the grid used here (steps of 1).

In [ ]:
chk = []
for m in range(1, 5):
    i = [j for j, s in enumerate(S) if s["mouse"] == m][0]
    s = S[i]
    rng = np.random.default_rng(7)
    X, y, _ = nj._xy(s, (1,), rng=rng)
    X = nj.zscore_bins(X)[:, nj.bin_index(s, -1)]
    I = s["type"] == "inhibitory"
    for pop, cols in (("all", slice(None)), ("inh", I)):
        a = nj.svm_cv(X[:, cols], y, np.random.default_rng(8), cgrid=nj.CGRID)["acc"]
        b = nj.svm_cv(X[:, cols], y, np.random.default_rng(8), cgrid=nj.CGRID_RUN)["acc"]
        chk.append({"mouse": m, "pop": pop, "full grid": a, "coarse grid": b, "difference": b - a})
print(pd.DataFrame(chk).round(2).to_string())

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-composition | reproduced |
| fig1b-psychometric | reproduced |
| figs1b-trial-history | blocked, data missing (no inter-trial intervals) |
| fig1gi-inferred-spiking-differs | partially reproduced |
| fig2a-example-session | not reproduced |
| fig2bc-fraction-selective-similar | not reproduced |
| fig2bd-selectivity-time-course | reproduced (assumption) |
| fig2de-selectivity-magnitude-similar | partially reproduced |
| figs3d-same-spiking-selectivity | reproduced |
| figs3c-wider-window | partially reproduced |
| figs3e-correct-error-correlation | partially reproduced |
| fig3bc-decoding-similar | reproduced |
| fig3b-decoding-time-course | partially reproduced |
| fig3df-weights-similar | reproduced |
| fig3hi-mixed-no-advantage | reproduced |
| figs5-fluorescence-decoding | blocked, data missing (no fluorescence traces) |
| figs6ab-previous-choice-stimulus | blocked, not reached in the run's time |
| figs6c-outcome-decoding | blocked, not reached in the run's time |
| figs6de-incorrect-trials | partially reproduced |
| fig4bd-stability-similar | reproduced |
| figs7c-angle-stability | blocked, too large |
| fig5a-nonselective-model, fig5bc-selective-models-match, figs9-signal-selective-best | blocked, not reached in the run's time (simulations) |
| fig6a-same-selectivity-correlations | reproduced |
| fig6bc-inhibitory-correlations-higher | reproduced (assumption) |
| fig7-noise-correlations-limit-decoding | reproduced (assumption) |
| fig8ab-learning-accuracy | not reproduced |
| fig8ce-choice-signal-onset | not reproduced |
| fig8de-learning-parallel | partially reproduced (assumption) |
| figs13c-equal-trials | partially reproduced |
| figs12-licking-running | blocked, metadata insufficient (lick spout not identified) |
| fig8f-correlations-decrease | partially reproduced |
| figs13ab-pseudo-learning | partially reproduced |
| fig8g-fraction-selective-triples | not reproduced |
| figs2-temporal-dynamics-similar | blocked, not reached in the run's time |
| figs14-neuropil, figs15-labeling-specificity | blocked, data missing (raw movies, histology) |

How the analysis differs from the paper's: the SVM regularisation is chosen from a grid with steps
of 1 in log10 (the paper 0.2; the check in `02_decoders.ipynb` shows differences of at most 0.67
points); excitatory, mixed and 2n-excitatory populations use 5 subsamples (3 in the learning
analysis) where the paper uses 50; the time courses use 25 splits per bin and one C per session
chosen at the bin before the choice; choices are matched once per session; and side labels are
"high-rate" and "low-rate" choices, as the files do not name the side.

What was found about the dataset: the session counts, trials per session and neuron counts match
the paper; the traces named `dFoF_*` hold the normalised inferred spiking activity, not Delta F/F,
and no fluorescence trace is in the files; the attribute named `rate` of each trial series holds the
frame interval; trial times are relative to each trial, so inter-trial intervals cannot be
computed; the lick trace does not say which spout was licked; some files store non-integer pulse
rates; and the NWB session identifiers are the CNMF run timestamps, while the recording date is in
`session_start_time` and `session_description`.


In [ ]:
print("runtime %.1f min, peak memory %.1f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))